### 实战：用 TodoListMiddleware 打造代码修复 Agent

本节把 `TodoListMiddleware` 用到实际场景：一个能**自动修改代码**的 Agent。它拥有四个工具：

| 工具 | 作用 |
| --- | --- |
| `list_files` | 列出工作区内的文件 |
| `read_file` | 读取文件内容 |
| `write_file` | 写入/覆盖文件 |
| `run_tests` | 运行单元测试 |

配合 `TodoListMiddleware`，Agent 会先规划待办（`write_todos`），再循环执行「读代码 → 定位 bug → 改代码 → 跑测试」，直到测试通过。

> 安全设计：所有文件操作都被限制在一个**临时工作区**内，路径越界会被拒绝，不会影响你的真实项目。

In [6]:
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

load_dotenv(override=True)

# 在当前目录下创建 temp 目录，并把临时工作区放在 temp/ 下面，避免影响真实项目
WORKSPACE_ROOT = Path.cwd() / "temp"
WORKSPACE_ROOT.mkdir(exist_ok=True)
WORKSPACE = Path(tempfile.mkdtemp(prefix="code_agent_", dir=WORKSPACE_ROOT)).resolve()
print("工作区：", WORKSPACE)

# 预置一个有 bug 的模块和对应测试，交给 Agent 修复
(WORKSPACE / "math_utils.py").write_text(
    'def add(a, b):\n'
    '    """返回两数之和。"""\n'
    '    return a - b\n'
    '\n'
    '\n'
    'def is_even(n):\n'
    '    """判断是否为偶数。"""\n'
    '    return n % 2 == 1\n',
    encoding="utf-8",
)

(WORKSPACE / "test_math_utils.py").write_text(
    'import unittest\n'
    '\n'
    'from math_utils import add, is_even\n'
    '\n'
    '\n'
    'class TestMathUtils(unittest.TestCase):\n'
    '    def test_add(self):\n'
    '        self.assertEqual(add(2, 3), 5)\n'
    '\n'
    '    def test_is_even(self):\n'
    '        self.assertTrue(is_even(4))\n'
    '        self.assertFalse(is_even(3))\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    unittest.main()\n',
    encoding="utf-8",
)

print("已预置 math_utils.py 与 test_math_utils.py")


工作区： /Users/lijixu/PycharmProjects/langchain_demo/charpter08-middleware/temp/code_agent_hcdze35o
已预置 math_utils.py 与 test_math_utils.py


#### 定义四个工具

In [7]:
def _resolve(path: str) -> Path:
    """把相对路径解析到工作区内；若越界则报错，防止读写工作区之外的文件。"""
    target = (WORKSPACE / path).resolve()
    if not target.is_relative_to(WORKSPACE):
        raise ValueError(f"路径越界，只允许在工作区内操作：{path}")
    return target


@tool
def list_files(path: Annotated[str, "相对工作区的目录，默认为 ."] = ".") -> str:
    """列出工作区内指定目录下的所有文件（返回相对路径）。"""
    base = _resolve(path)
    if not base.exists():
        return f"目录不存在：{path}"
    files = sorted(str(p.relative_to(WORKSPACE)) for p in base.rglob("*") if p.is_file())
    return "\n".join(files) if files else "（空）"


@tool
def read_file(path: Annotated[str, "相对工作区的文件路径"]) -> str:
    """读取工作区内指定文件的完整内容。"""
    target = _resolve(path)
    if not target.is_file():
        return f"文件不存在：{path}"
    return target.read_text(encoding="utf-8")


@tool
def write_file(
    path: Annotated[str, "相对工作区的文件路径"],
    content: Annotated[str, "要写入的完整文件内容"],
) -> str:
    """把内容写入工作区内的文件（覆盖原文件，父目录会自动创建）。"""
    target = _resolve(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"已写入 {path}（{len(content)} 字符）"


@tool
def run_tests(path: Annotated[str, "测试目录或文件，默认为 ."] = ".") -> str:
    """在工作区运行单元测试，返回退出码和输出（有 pytest 用 pytest，否则用 unittest）。"""
    target = _resolve(path)
    if importlib.util.find_spec("pytest") is not None:
        cmd = [sys.executable, "-m", "pytest", "-q", str(target)]
    else:
        cmd = [sys.executable, "-m", "unittest", "discover", "-s", str(target), "-v"]
    proc = subprocess.run(cmd, cwd=WORKSPACE, capture_output=True, text=True)
    return f"退出码：{proc.returncode}\n{proc.stdout}\n{proc.stderr}"


#### 创建 Agent

In [8]:
model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)

agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file, run_tests],
    # TodoListMiddleware 提供 write_todos 工具和 todos 状态，让 Agent 先规划再执行
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "你是一个代码修复助手：请先规划待办事项，再阅读代码、定位并修复 bug，"
        "最后运行测试确认全部通过。"
    ),
)


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


#### 运行：让 Agent 自动修复 bug

In [9]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "工作区里有 bug。请先用 write_todos 列出修复计划，"
                    "然后阅读代码、定位并修复 bug，最后运行测试确认全部通过。"
                ),
            }
        ]
    }
)

# 1) Agent 的待办规划
print("=== 待办事项 ===")
for item in result.get("todos", []):
    print(f"  [{item['status']}] {item['content']}")

# 2) Agent 的完整工具调用轨迹
print("\n=== 工具调用轨迹 ===")
for message in result["messages"]:
    for tool_call in getattr(message, "tool_calls", None) or []:
        args = str(tool_call["args"]).replace("\n", " ")
        print(f"  {tool_call['name']}({args[:70]})")

# 3) 最终回答
print("\n=== 最终回答 ===")
print(result["messages"][-1].content)


=== 待办事项 ===
  [completed] Explore workspace structure and identify source/test files
  [completed] Read the code and tests to locate the bug
  [completed] Fix the bug
  [completed] Run tests to confirm all pass

=== 工具调用轨迹 ===
  write_todos({'todos': [{'content': 'Explore workspace structure and identify sourc)
  list_files({'path': '.'})
  read_file({'path': 'math_utils.py'})
  read_file({'path': 'test_math_utils.py'})
  write_todos({'todos': [{'content': 'Explore workspace structure and identify sourc)
  write_file({'path': 'math_utils.py', 'content': 'def add(a, b):\n    """返回两数之和。"")
  run_tests({'path': '.'})
  write_todos({'todos': [{'content': 'Explore workspace structure and identify sourc)

=== 最终回答 ===
在 `math_utils.py` 中发现并修复了两个 bug：

1. **`add(a, b)`** — 返回的是 `a - b`（减法）而不是和。已改为 `return a + b`。
2. **`is_even(n)`** — 用 `n % 2 == 1` 判断偶数，这实际上是在判断奇数。已改为 `return n % 2 == 0`。

测试结果：**2 passed**（退出码 0），全部通过。


#### 检查修复结果

In [10]:
# 直接查看修复后的文件，并再次运行测试确认
print("=== 修复后的 math_utils.py ===")
print((WORKSPACE / "math_utils.py").read_text(encoding="utf-8"))

print("=== 重新运行测试 ===")
print(run_tests.invoke({"path": "."}))


=== 修复后的 math_utils.py ===
def add(a, b):
    """返回两数之和。"""
    return a + b


def is_even(n):
    """判断是否为偶数。"""
    return n % 2 == 0

=== 重新运行测试 ===
退出码：0
..                                                                       [100%]
2 passed in 0.00s




#### 要点回顾

1. `TodoListMiddleware` 让 Agent 在动手前先规划，适合代码修改这类多步骤任务。
2. 四个工具各司其职：`list_files` / `read_file` / `write_file` / `run_tests`，工具要返回**可读的结果**（例如测试的退出码与输出），Agent 才能据此决策。
3. 生产环境中，文件操作一定要做**路径限制**（如 `_resolve`）避免越权。
4. 可进一步叠加 `HumanInTheLoopMiddleware`，让 `write_file` 等写操作在执行前经人工确认；或叠加 `PIIMiddleware` 对内容脱敏。